- Qué registros tienen nulos? En los logs se ve por qué?
- Qué registros tienen valores atípicos y cómo afecta eso a las pruebas estadísticas?
- Ver si muchos tienen un valor muy similar y muy alto como los que les aparecía el error

Qué registros tienen nulos o infinitos? Cómo se ven con respecto al de IGD+ que es el que parece funcionar mejor?

In [1]:
from tqdm.notebook import tqdm as tqm
import matplotlib.pyplot as plt
import plotly.graph_objs as go
from utils.func_aux import *
from utils.func_vis import *
import plotly.express as px
import scipy.stats as st
import seaborn as sns
import pandas as pd
import numpy as np
import datetime
import shutil
import time 
import os

# Opciones de matplotlib
rc = plt.rcParams
rc["figure.figsize"] = [15, 5]

# Para mostrar todas las columnas cuando se imprime un df
pd.set_option("display.max_columns", None)

# Para poner el estilo de las gráficas de matplotlib parecido al de ggplot
plt.style.use("ggplot") 

go_to_PCUI_Proyect()

In [65]:
tablas_indicadores = {
    'igd+':pd.read_csv('../../tablas_generadas/todos_QI_IGDP_20_runs_senergy.csv'),
    'eps+':pd.read_csv('../../tablas_generadas/todos_QI_EPSP_codigo_comentado_senergy.csv'),
    'r2':pd.read_csv('../../tablas_generadas/todos_QI_R2_codigo_comentado_senergy.csv')
                      }

Vemos que todas las tablas tengan el mismo número de indicadores

In [66]:
for indicador, tabla_indicador in tablas_indicadores.items():
    print(len(tabla_indicador), indicador) 

147840 igd+
147840 eps+
147840 r2


Unimos las tablas

In [67]:
import pandas as pd
import pandas as pd
df_lista = []

for indicador, tabla_indicador in tablas_indicadores.items():
    tabla_indicador['hiperparam_ind_conv']=indicador.upper()
    tabla_indicador.rename(columns={'dimension':'n_objetivos','w0':'w_0','valor':'valor_indicador'}, inplace=True)
    tabla_indicador = tabla_indicador[['hiperparam_ind_conv','problema','n_objetivos','w_0','run','indicador','valor_indicador']]
    df_lista.append(tabla_indicador)


df_lista = pd.concat(df_lista, axis=0, ignore_index=True)
df_lista['w_0'] = df_lista['w_0'].replace({w_str:[0.001, 0.1  , 0.2  , 0.3  , 0.4  , 0.5  , 0.6  , 0.7  , 0.8  ,
       0.9  , 0.999][i] for i,w_str in enumerate(df_lista['w_0'].unique())})
df_lista.to_csv('QI_R2_EPSP_IGD+.csv',index=False)


In [72]:
df_lista.to_csv('../../tablas_generadas/QI_R2_EPSP_IGD+.csv',index=False)


In [71]:
df_lista.query('(problema=="WFG9") & (n_objetivos==3) &(w_0==0.001) & (run==0) & (indicador=="eps+")')

,hiperparam_ind_conv,problema,n_objetivos,w_0,run,indicador,valor_indicador
0,IGD+,WFG9,3,0.001,0,eps+,0.217718
147840,EPS+,WFG9,3,0.001,0,eps+,0.179001
295680,R2,WFG9,3,0.001,0,eps+,0.221069


In [60]:
get_sol_path(path='../../archivos_EPS+_codigo_comentado/w_03/PCUIEMOA_DTLZ1_07D_R08.pof',n_objetivos=7)

,f0,f1,f2,f3,f4,f5,f6
0,0.090182,0.001681,0.200143,0.001346,0.000000,0.089337,0.121425
1,0.000000,0.000000,0.000000,0.121586,0.267571,0.123000,0.000454
2,0.005839,0.409104,0.000950,0.186739,0.056371,0.003803,0.149443
3,0.000000,0.000000,0.412591,0.000000,0.000000,0.000883,0.090800
4,0.380642,0.002549,0.000000,0.000742,0.003878,0.116623,0.000099
...,...,...,...,...,...,...,...
115,0.112117,0.002405,0.000924,0.290455,0.002735,0.000823,0.095463
116,0.000000,0.000000,0.000000,0.000000,0.000000,0.333927,0.170223
117,0.000000,0.000000,0.000000,0.275171,0.000000,0.228492,0.000461
118,0.176273,0.134913,0.190293,0.000000,0.000000,0.003057,0.000002


In [58]:
df_lista[df_lista['valor_indicador'].astype(float)==np.inf].query('problema=="DTLZ3"')

,hiperparam_ind_conv,problema,n_objetivos,w_0,run,indicador,valor_indicador
26248,IGD+,DTLZ3,7,0.1,8,s-energy,inf
53128,IGD+,DTLZ3,7,0.3,8,s-energy,inf
80008,IGD+,DTLZ3,7,0.5,8,s-energy,inf
80013,IGD+,DTLZ3,7,0.5,13,s-energy,inf
120334,IGD+,DTLZ3,7,0.8,14,s-energy,inf
200960,EPS+,DTLZ3,7,0.3,0,s-energy,inf
200976,EPS+,DTLZ3,7,0.3,16,s-energy,inf
254723,EPS+,DTLZ3,7,0.7,3,s-energy,inf
254738,EPS+,DTLZ3,7,0.7,18,s-energy,inf
321928,R2,DTLZ3,7,0.1,8,s-energy,inf


In [69]:
df_lista[df_lista['valor_indicador'].astype(float)==np.inf].groupby(['hiperparam_ind_conv','problema','indicador','n_objetivos','w_0']).size().reset_index(name='count')

,hiperparam_ind_conv,problema,indicador,n_objetivos,w_0,count
0,EPS+,DTLZ3,s-energy,7,0.300,2
1,EPS+,DTLZ3,s-energy,7,0.700,2
2,EPS+,DTLZ4,s-energy,2,0.001,1
3,EPS+,DTLZ4,s-energy,2,0.200,1
4,EPS+,DTLZ4,s-energy,2,0.300,1
5,EPS+,DTLZ4,s-energy,2,0.500,1
6,EPS+,DTLZ4,s-energy,2,0.999,1
7,EPS+,DTLZ4,s-energy,7,0.001,2
8,EPS+,DTLZ4,s-energy,7,0.100,2
9,EPS+,DTLZ4,s-energy,7,0.200,2


In [70]:
df_lista[np.isclose(df_lista['valor_indicador'].astype(float),0)].groupby(['hiperparam_ind_conv','problema','indicador','n_objetivos','w_0']).size().reset_index(name='count')#['problema'].unique()

,hiperparam_ind_conv,problema,indicador,n_objetivos,w_0,count
0,EPS+,DTLZ3,hv,7,0.001,9
1,EPS+,DTLZ3,hv,7,0.100,7
2,EPS+,DTLZ3,hv,7,0.200,7
3,EPS+,DTLZ3,hv,7,0.300,5
4,EPS+,DTLZ3,hv,7,0.400,5
...,...,...,...,...,...,...
329,R2,DTLZ6,hv,7,0.600,20
330,R2,DTLZ6,hv,7,0.700,20
331,R2,DTLZ6,hv,7,0.800,20
332,R2,DTLZ6,hv,7,0.900,20


In [22]:
pd.read_csv('../../tablas_generadas/todos_QI.csv')#['w_0'].unique()

,hiperparam_ind_conv,problema,n_objetivos,w_0,run,indicador,valor_indicador
0,IGD+,WFG9,3,0.001,0,eps+,0.217718
1,IGD+,WFG9,3,0.001,1,eps+,0.240986
2,IGD+,WFG9,3,0.001,2,eps+,0.240616
3,IGD+,WFG9,3,0.001,3,eps+,0.236098
4,IGD+,WFG9,3,0.001,4,eps+,0.267390
...,...,...,...,...,...,...,...
147835,R2,WFG9,6,0.999,5,hv,93347.990000
147836,R2,WFG9,6,0.999,6,hv,97336.020000
147837,R2,WFG9,6,0.999,7,hv,95928.310000
147838,R2,WFG9,6,0.999,8,hv,98113.420000


Hace

Vemos que tengan valores distintos (i.e. que no se copiaron valores similares) y que corresponden a los archivos dentro de los folders correspondientes

Buscar el DTLZ3 en 7 objetivos con 

DTLZ3 7 16 4 (0.001, 0.999)
Ejecutando comando
./pcuiemoa input/Param_07D-0.cfg DTLZ3 20 augmented_chebyshev_pcui EPS+ ES

Ver si este tiene un pésimo valor de indicadores